In [3]:
import pandas as pd

# Replace 'your_file.data' with the actual path to your .data file
# If your file has a different delimiter (e.g., space, tab), change `delimiter` accordingly.
# If your file has headers, remove `header=None`.
# If your file has no headers and you know the column names, you can provide them in a list to `names` argument.
file_path = 'adult.data'
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']
df = pd.read_csv(file_path, delimiter=',', header=None, names=column_names)
df

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


In [4]:
df.shape

(32561, 15)

In [5]:
df.describe()

,age,fnlwgt,education_num,capital_gain,capital_loss,hours_per_week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


### Model Training and Evaluation

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

### Data Preprocessing

In [7]:
# Replace ' ?' values with NaN, which are common in this dataset for missing values
df_processed = df.replace(' ?', np.nan)

# Identify categorical and numerical features
categorical_features = df_processed.select_dtypes(include=['object']).columns.tolist()
numerical_features = df_processed.select_dtypes(include=np.number).columns.tolist()

# Remove 'income' from categorical features as it's the target variable
if 'income' in categorical_features:
    categorical_features.remove('income')

print(f"Categorical features: {categorical_features}")
print(f"Numerical features: {numerical_features}")

# Handle missing values: For simplicity, fill numerical NaNs with the mean and categorical NaNs with the most frequent value.
for col in numerical_features:
    if df_processed[col].isnull().any():
        df_processed[col] = df_processed[col].fillna(df_processed[col].mean())

for col in categorical_features:
    if df_processed[col].isnull().any():
        df_processed[col] = df_processed[col].fillna(df_processed[col].mode()[0])

# Check for remaining missing values
print("\nMissing values after initial handling:\n", df_processed.isnull().sum())

Categorical features: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']
Numerical features: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']

Missing values after initial handling:
 age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


### Encode Target Variable

In [8]:
#encode the target variable 'income.' <=50K will be 0, >50K will be 1
le = LabelEncoder()
df_processed['income_encoded'] = le.fit_transform(df_processed['income'])

print("Original income categories:", le.classes_)
print("Encoded income unique values:", df_processed['income_encoded'].unique())

Original income categories: [' <=50K' ' >50K']
Encoded income unique values: [0 1]


### One-Hot Encode Categorical Features and Split Data

In [9]:
X = df_processed.drop(['income', 'income_encoded'], axis=1)
y = df_processed['income_encoded']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)

X_processed = preprocessor.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42, stratify=y)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (26048, 105)
Shape of X_test: (6513, 105)
Shape of y_train: (26048,)
Shape of y_test: (6513,)


In [10]:
log_reg = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

print(f"Best hyperparameters: {grid_search.best_params_}")

print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

best_log_reg = grid_search.best_estimator_

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best hyperparameters: {'C': 1, 'penalty': 'l1'}
Best cross-validation accuracy: 0.8496


###Tune Hyperparameters and Train the Model

In [11]:
log_reg = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)

param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2']
}

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(estimator=log_reg, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

print(f"Best hyperparameters: {grid_search.best_params_}")

print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

best_log_reg = grid_search.best_estimator_

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best hyperparameters: {'C': 1, 'penalty': 'l1'}
Best cross-validation accuracy: 0.8496


In [12]:

y_pred = best_log_reg.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.4f}")
print("\nClassification Report:\n", report)
print("\nConfusion Matrix:\n", conf_matrix)

Test Accuracy: 0.8552

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.93      0.91      4945
           1       0.74      0.62      0.67      1568

    accuracy                           0.86      6513
   macro avg       0.81      0.77      0.79      6513
weighted avg       0.85      0.86      0.85      6513


Confusion Matrix:
 [[4605  340]
 [ 603  965]]


### Displaying Predicted Results

In [13]:
import pandas as pd

print("First 10 Predicted Values (encoded):", y_pred[:10])
print("First 10 Actual Values (encoded):  ", y_test[:10].values)

results_df = pd.DataFrame({'Actual_Encoded': y_test, 'Predicted_Encoded': y_pred})

results_df = results_df.reset_index(drop=True)

results_df['Actual_Income'] = le.inverse_transform(results_df['Actual_Encoded'])
results_df['Predicted_Income'] = le.inverse_transform(results_df['Predicted_Encoded'])

print("\nSample of Actual vs. Predicted Income:")
display(results_df.head(10))

First 10 Predicted Values (encoded): [0 0 1 1 0 0 0 1 0 0]
First 10 Actual Values (encoded):   [0 0 1 1 0 0 0 0 0 0]

Sample of Actual vs. Predicted Income:


,Actual_Encoded,Predicted_Encoded,Actual_Income,Predicted_Income
0,0,0,<=50K,<=50K
1,0,0,<=50K,<=50K
2,1,1,>50K,>50K
3,1,1,>50K,>50K
4,0,0,<=50K,<=50K
5,0,0,<=50K,<=50K
6,0,0,<=50K,<=50K
7,0,1,<=50K,>50K
8,0,0,<=50K,<=50K
9,0,0,<=50K,<=50K


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

grid_search_rf = GridSearchCV(
    estimator=rf,
    param_grid=param_grid_rf,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search_rf.fit(X_train, y_train)

print(f"Best RF hyperparameters: {grid_search_rf.best_params_}")
print(f"Best RF cross-validation accuracy: {grid_search_rf.best_score_:.4f}")

best_rf = grid_search_rf.best_estimator_

Fitting 5 folds for each of 12 candidates, totalling 60 fits


## Predict and Evaluate Model Performance

In [ ]:
rf_pred = best_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)
rf_report = classification_report(y_test, rf_pred)
rf_conf_matrix = confusion_matrix(y_test, rf_pred)

print(f"Random Forest Test Accuracy: {rf_accuracy:.4f}")
print("\nRandom Forest Classification Report:\n", rf_report)
print("\nRandom Forest Confusion Matrix:\n", rf_conf_matrix)

# Compare Actual vs Predicted Results

In [ ]:
rf_results_df = pd.DataFrame({
    'Actual_Encoded': y_test,
    'Predicted_Encoded': rf_pred
})

rf_results_df = rf_results_df.reset_index(drop=True)

rf_results_df['Actual_Income'] = le.inverse_transform(rf_results_df['Actual_Encoded'])
rf_results_df['Predicted_Income'] = le.inverse_transform(rf_results_df['Predicted_Encoded'])

print("\nSample of Actual vs. Predicted Income (Random Forest):")
display(rf_results_df.head(10))

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [accuracy, rf_accuracy]
})

print(comparison_df)